# 1. 지역별 데이터 불러오기

In [ ]:
import pandas as pd
import os
import re

# 폴더 경로 설정
folder_path = '../Database/opendata'

# csv 파일 필터링
file_list = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

# 데이터프레임 리스트
df_list = []

for file in file_list:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(file_path, encoding='utf-8-sig') 
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, encoding='euc-kr')  
    df_list.append(df)

# 모든 데이터를 하나로 합치기
all_data = pd.concat(df_list, ignore_index=True)

# 미리보기
all_data.head()

# 총 데이터 개수 확인: (412178, 35)

all_data.shape
col_names = all_data.columns

In [5]:
import pandas as pd

df = pd.read_csv('Database/opendata/경상남_ev_charger_info.csv')

/var/folders/xh/q3k02qfx1_n8llwstgwvcj900000gn/T/ipykernel_55370/2571415473.py:3: DtypeWarning: Columns (31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Database/opendata/경상남_ev_charger_info.csv')


# 데이터 전처리

## 2. 데이터 전처리(필수)
- kindDetail에서 아파트 등 접근 불가한 데이터 지우기
- delYn 변수에서 n인 것들만 사용
    - 충전기 정보라서 충전소 기준으로 살펴봐야함
- useTime 24시간인지 아닌지 컬럼 전처리
- 시간 관련 column 데이터타입 변경

- kindDetail

In [13]:
# 외부 접근 자유로운 장소
accessible_kinds = [
    "A001", "A002", "A003", "A004",     # 공공시설, 지자체
    "B001", "B002", "B003", "B004",     # 공영/공원/일반 주차장
    "C001", "C002", "C003",             # 휴게소, 쉼터
    "D001", "D002", "D004", "D005",     # 공원, 전시관, 생태공원, 홍보관
    "D006", "D007", "D008", "D009",     # 관광안내소, 관광지, 박물관, 유적지
    "J004", "J005", "J006", "J007"      # 공연장, 관람장, 동식물원, 경기장
]
# 접근은 가능하나 비용/제약 가능성 높은 곳
costly_kinds = [
    "E001", "E002", "E003",             # 마트, 백화점, 숙박시설
    "E004", "E005", "E006",             # 골프장, 카페, 음식점
    "E008",                             # 영화관
    "I001", "I002", "I003",             # 병원, 종교시설, 보건소
    "I004", "I005", "I006", "I008",     # 경찰서, 도서관, 복지관, 금융기관
    "J001", "J002", "J003"              # 학교, 교육원, 학원
]
# 외부 접근 불가 또는 일반 사용 제한
inaccessible_kinds = [
    "F001", "F002",                     # 정비소, 서비스센터 (내부 고객 전용 가능성)
    "G001", "G005", "G006",             # 군부대, 오피스텔, 단독주택
    "H001", "H002", "H003", "H004",     # 아파트, 빌라, 사옥, 기숙사
    "H005",                             # 연립주택
    "I007",                             # 수련원
    "J001", "J002"                      # 학교, 교육원
]
# 충전만 가능한 특수 구조 (잠시 정차 가능)
charging_only_kinds = [
    "E007",                             # 주유소
    "G003", "G004"                      # 공중전화부스, 기타
]

- kind

In [14]:
# ✅ 외부 접근 자유로운 시설
accessible = [
    "A0",  # 공공시설
    "B0",  # 주차시설
    "C0",  # 휴게시설
    "D0"   # 관광시설
]

# ⚠️ 접근은 가능하나 비용/제약 가능성 있는 시설
costly = [
    "E0",  # 상업시설 (마트, 음식점, 숙박 등)
    "I0",  # 근린생활시설 (병원, 도서관 등)
    "J0"   # 교육문화시설 (학교, 학원 등)
]

# ❌ 외부 접근이 제한되거나 불가능한 시설
inaccessible = [
    "F0",  # 차량정비시설
    "G0",  # 기타시설 (오피스텔, 주택 포함 가능성)
    "H0"   # 공동주택시설
]

- delYn

In [55]:
# 삭제된 충전소(운영하지 않는 충전소) 제거
df = df[df['delYn']=='N']

- useTime

In [33]:
# 1. 아파트, 접근 불가 충전소 제거
df = df[~df['kindDetail'].isin(inaccessible_kinds)]

# 2. 삭제된 충전소 제외
df = df[df['delYn'] == 'N']

# 3. 운영시간 정보 전처리
def clean_use_time(val):
    if pd.isna(val): return "제한"
    val = str(val)
    if "24시" in val: return "24시간"
    elif "00:00 ~ 24:00" in val: return "24시간"
    elif "0000~0000" in val: return "24시간"
    elif "00:00 ~ 23:59" in val: return "24시간"
    # 주중, 주말, 월~금, 공휴일, 토, 일, 
    elif "외부인" in val: return "제한"
    elif "제한" in val: return "제한"
    elif "~" in val: return "제한"
    else: return "기타"

df['useTimeCleaned'] = df['useTime'].apply(clean_use_time)

In [34]:
df

,statNm,statId,chgerId,chgerType,addr,addrDetail,location,useTime,lat,lng,...,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName,useTimeCleaned
15,한국도로공사창원지사,ME178131,1,6,경상남도 창원시 의창구 의창대로 1024,NaN,NaN,24시간 이용가능,35.284613,128.700209,...,Y,NaN,N,NaN,N,NaN,N,2017,경상남도,24시간
20,한려해상국립공원사무소 노량분소,ME181203,1,6,경상남도 남해군 설천면 노량리 410-36,NaN,NaN,24시간 이용가능,34.939704,127.875309,...,Y,NaN,N,NaN,N,NaN,N,2018,경상남도,24시간
21,한려해상국립공원사무소 노량분소,ME181203,2,6,경상남도 남해군 설천면 노량리 410-36,NaN,NaN,24시간 이용가능,34.939704,127.875309,...,Y,NaN,N,NaN,N,NaN,N,2018,경상남도,24시간
22,한려해상국립공원사무소 금산분소,ME181204,1,6,경상남도 남해군 이동면 보리암로 287,NaN,NaN,24시간 이용가능,34.773405,127.987807,...,Y,NaN,N,NaN,N,NaN,N,2018,경상남도,24시간
23,한려해상국립공원사무소 금산분소,ME181204,2,6,경상남도 남해군 이동면 보리암로 287,NaN,NaN,24시간 이용가능,34.773405,127.987807,...,Y,NaN,N,NaN,N,NaN,N,2018,경상남도,24시간
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25150,비앤비골프클럽,TS482501,1,2,경상남도 김해시 진영읍 진영산복로 130-21,NaN,NaN,24시간 이용가능,35.300407,128.732027,...,Y,NaN,N,NaN,N,NaN,N,2025,경상남도,24시간
25151,비앤비골프클럽,TS482501,2,2,경상남도 김해시 진영읍 진영산복로 130-21,NaN,NaN,24시간 이용가능,35.300407,128.732027,...,Y,NaN,N,NaN,N,NaN,N,2025,경상남도,24시간
25159,클럽디 거창,TUID0003,1,2,경상남도 거창군 신원면 감악산로 398,클럽디 거창,클럽하우스 오른편 주차장,24시간 이용가능,35.592252,127.905408,...,Y,NaN,N,NaN,N,NaN,N,2023,경상남도,24시간
25160,클럽디 거창,TUID0003,2,2,경상남도 거창군 신원면 감악산로 398,클럽디 거창,클럽하우스 오른편 주차장,24시간 이용가능,35.592252,127.905408,...,Y,NaN,N,NaN,N,NaN,N,2023,경상남도,24시간


In [35]:
df_ex = df[df["useTimeCleaned"] == "24시간"]
df_ex['useTime'].unique()

array(['24시간 이용가능', '월~금24시간 이용가능', '24시 개방', '24시간 개방', '0000~0000',
       '24시간 이용가능(투숙객 외 이용불가)', '24시간 이용가능(외부인출입불가)',
       '24시간 이용가능(시설상황에 따라 이용제한가능)', '24시간 이용가능(시설이용자 외 이용제한 있을 수 있음)',
       '24시간 이용가능/시설 상황에 따라 이용이 제한될 수 있습니다',
       '24시간 이용가능/시설 상황에 따라 이용이 제한될 수', '24시간 이용', '24시간이용',
       '24시간 이용가능 (19:00~22:00 무료)', '00:00 ~ 23:59', '주중/주말 : 24시간',
       '24시간이용가능', '24시간', '비공용충전기(24시)', '24시간 이용시간', '00:00 ~ 24:00'],
      dtype=object)

In [36]:
df_ex2 = df[df["useTimeCleaned"] == "제한"]
df_ex2['useTime'].unique()

array(['10:00~23:00', '이용제한시간 (06:00~07:30) 외 이용가능', '10:00~24:00',
       '09:00~18:00', '09:00~21:00', '09:00~19:00', '10:00 ~ 23:00',
       '09:30 ~ 23:00', '화~일요일 09:00~18:00 이용가능 (월요일 이용불가)',
       '09:00 ~ 18:00 개방', '평일 09:00 ~ 18:00, 주말미개방', '07:00 ~ 19:00',
       '09:00 ~ 18:00', '(월~금)09:00 ~ 18:00, 토/일 및 휴무일 사용불가', '9시~18시',
       '9:00~18:00', '09시 ~ 18시', '05시 ~ 20시', '(평일) 09시00분 ~ 18시00분',
       '~', '외부인 사용불가', '09:00~18:00(시설상황에 따라 이용제한가능)', '09:00~22:00',
       '09:00 ~ 22:00', '05:00~22:00', '11:30~21:00', '10:00~22:00',
       '평일 09:00~18:00', '평일 08:00~17:00', '평일 09시~18시 이용가능',
       '평일 09시~18시 이용가능, 주말/공휴일 이용불가', '월~금 07:00~20:00, 주말 08:00~20:00',
       '출입에 제한 있음', nan, '주중 08:00~19:00, 주말 09:00~18:00',
       '10:30~20:00(금토일/공휴일 20:30)', '10:00 ~ 22:00', '09:00 ~ 24:00',
       '10:00 ~ 24:00', '07:00 ~ 22:00', '09:30 ~ 24:00',
       '관계자전용 09:00~18:00', '매장운영시간(10시~22시)', '주중/주말 : 06시~23시',
       '06~23시', '9:00 ~ 21:00', '07:00~22:00', '10:00 ~ 1

In [37]:
df_ex3 = df[df["useTimeCleaned"] == "기타"]
df_ex3['useTime'].unique()

array(['08시-19시', '1시간 무료, 1시간 초과시 주차요금 50% 경감', '07:00-22:00',
       '건물 리모델링으로 사용불가', '공사중-이용불가', '10:00-22:00',
       '비숙박자일시 PM-15:00까지 사용가능', '미개방', '마트운영시간', '매장영업시간', '매장운영시간',
       '업소고객전용', '비개방(업소고객전용)', '비개방(임직원전용)', '건물측 주차 관리 시설로 문의', '상시',
       '이마트 운영시간 내'], dtype=object)

- useTime llm 프롬프팅으로 데이터 정제

In [ ]:
import openai
import pandas as pd
import json
import time

openai.api_key = ""  # 🔐 본인 키로 교체

In [51]:
def parse_use_time_with_label(use_time_text: str) -> dict:
    if pd.isna(use_time_text) or use_time_text.strip() == "":
        return {"label": "unknown", "hours": None}

    prompt = f"""
다음 운영시간 정보를 기반으로, 운영시간 분류 label과 요일별 개장 시간을 구조화해주세요.

출력은 반드시 다음과 같은 JSON 형식이어야 합니다:
{{
  "label": "fulltime" 또는 "limited" 또는 "restricted" 또는 "unknown",
  "hours": {{
    "monday": ["09:00", "18:00"] 또는 null,
    ...
    "sunday": ...
  }}
}}

다음은 예시입니다.

입력: "평일 09:00 ~ 18:00, 주말미개방"
출력:
{{
  "label": "limited",
  "hours": {{
    "monday": ["09:00", "18:00"],
    "tuesday": ["09:00", "18:00"],
    "wednesday": ["09:00", "18:00"],
    "thursday": ["09:00", "18:00"],
    "friday": ["09:00", "18:00"],
    "saturday": null,
    "sunday": null
  }}
}}

입력: "24시간 이용가능"
출력:
{{
  "label": "fulltime",
  "hours": {{
    "monday": ["00:00", "24:00"],
    "tuesday": ["00:00", "24:00"],
    "wednesday": ["00:00", "24:00"],
    "thursday": ["00:00", "24:00"],
    "friday": ["00:00", "24:00"],
    "saturday": ["00:00", "24:00"],
    "sunday": ["00:00", "24:00"]
  }}
}}

입력: "{use_time_text}"
출력:
    """

    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        content = response["choices"][0]["message"]["content"]
        return json.loads(content)
    except Exception as e:
        print(f"⚠️ 오류 발생: {e}")
        return {"label": "unknown", "hours": None}

In [52]:
# 중복 제거하고 정리된 목록만 추출
unique_use_times = df['useTime'].dropna().unique()[:3]
unique_use_times

array(['24시간 이용가능', '10:00~23:00', '이용제한시간 (06:00~07:30) 외 이용가능'],
      dtype=object)

In [53]:
# 결과 저장 딕셔너리
parsed_times = {}

# 하나씩 LLM으로 호출 (캐싱 겸용)
for text in unique_use_times:
    print(f"🔍 처리 중: {text}")
    parsed_times[text] = parse_use_time_with_label(text)
    time.sleep(1)  # LLM 과금, 속도 제한 고려 → 너무 빠르게 돌리면 에러

# 결과 확인 (샘플)
print(parsed_times["평일 09:00 ~ 18:00, 주말미개방"])

🔍 처리 중: 24시간 이용가능
⚠️ 오류 발생: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

🔍 처리 중: 10:00~23:00
⚠️ 오류 발생: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742

🔍 처리 중: 

KeyError: '평일 09:00 ~ 18:00, 주말미개방'

-  시간 관련 컬럼 statUpdDt, lastTsdt, lastTedt, nowTsdt

In [54]:
# 시간 관련 칼럼 데이터타입 변경 

for col in ['statUpdDt', 'lastTsdt', 'lastTedt', 'nowTsdt']:
    df.loc[:, col] = pd.to_datetime(df[col], format='%Y%m%d%H%M%S', errors='coerce')

/var/folders/xh/q3k02qfx1_n8llwstgwvcj900000gn/T/ipykernel_55370/2351843780.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<DatetimeArray>
['2025-05-15 15:57:03', '2025-05-15 15:54:30', '2025-05-15 15:53:07',
 '2025-05-15 15:53:24', '2025-05-15 15:53:11', '2025-05-15 15:55:11',
 '2025-01-17 13:30:00', '2025-05-15 15:53:32', '2025-05-15 15:53:23',
 '2025-05-15 15:56:37',
 ...
 '2025-05-03 18:15:05', '2025-05-02 09:47:03', '2025-04-29 14:03:03',
 '2025-05-14 14:21:04', '2025-05-11 14:02:03', '2025-04-29 08:50:56',
 '2025-04-29 08:50:56', '2025-05-14 12:31:01', '2025-05-14 11:43:23',
 '2025-05-12 11:23:10']
Length: 7411, dtype: datetime64[ns]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[:, col] = pd.to_datetime(df[col], format='%Y%m%d%H%M%S', errors='coerce')
/var/folders/xh/q3k02qfx1_n8llwstgwvcj900000gn/T/ipykernel_55370/2351843780.py:4: FutureWarning: Setti

## 3. 로직 1 - 1차 필터링

로직 1: 사용자 입력 변수를 기반으로 한 필터링

1차 필터링: 고장나지 않은 충전기만 필터링
- 이용자제한 여부 -> limitYn 기준으로 이용자 제한 시설 제외 -> 고민 필요
- stat 변수에서 1, 2, 3, 9번만 사용 (1: 통신이상, 2: 충전대기, 3: 충전중, 9: 상태미확인)
- 고장/삭제 여부: 최근 사용이 현재 시점으로부터 48시간 이내에 발생한 적이 있는 충전소는 정상 충전소로 간주 

In [ ]:
# limitYn으로 거르기엔 그냥 단순 시간 제한이 사유인 곳도 많아서 고민. 제한 사유가 '제한 없음' 인 곳도 있고..

all_data_4[all_data_4['limitYn']=='Y']['limitDetail'].unique()[43:60]

array(['버스전용', '제한없음', '제한 없음', '.', '마트(쇼핑몰) 이용자, 상가 입주자로 사용 제한',
       '택시차고지 차량이 다수로 이용이 어려울수 있음', '전기버스 전용(DC콤보2)',
       '시설 사용자 외 이용제한 있을 수 있음', '외부인이 전기차 충전목적만으로 출입할 수 없습니다', '외부인 출입불가',
       '관용차량 충전 전용', '구내시설 상황에 따라 이용이 제한될 수 있음',
       '시설 상황에 따라 이용이 제한될 수 있음,', '학교 교직원으로 제한', '숙박객외 사용불가',
       '매장/시설 이용고객만 사용가능', '직원 및 방문자 전용'], dtype=object)

In [ ]:
# stat이 1인 경우에도 정상 작동하는 경우가 있는지 확인 -> 일단 48시간 이내에 사용된 기록이 있는 곳도 있긴 해서 포함. 

all_data_4[all_data_4['stat']==1].iloc[:, 10:].head(5)

,busiId,bnm,busiNm,busiCall,stat,statUpdDt,lastTsdt,lastTedt,nowTsdt,powerType,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
874,CG,서울씨엔지,서울씨엔지(서울이브이),1566-6373,1,2025-05-14 23:47:03,2025-05-14 21:50:31,2025-05-14 23:10:22,NaT,NaN,...,J007,Y,NaN,N,NaN,N,NaN,N,2023,강원특별자치도
875,CG,서울씨엔지,서울씨엔지(서울이브이),1566-6373,1,2025-05-14 23:47:03,2025-05-14 21:50:31,2025-05-14 23:10:22,NaT,NaN,...,J007,Y,NaN,N,NaN,N,NaN,N,2023,강원특별자치도
958,CI,쿨사인,쿨사인 주식회사,1600-4045,1,2025-04-30 06:59:41,2024-11-11 17:24:35,2024-11-11 17:30:02,NaT,NaN,...,G004,Y,NaN,N,NaN,N,NaN,N,2024,강원특별자치도
959,CI,쿨사인,쿨사인,1600-4045,1,2025-05-07 09:30:04,2025-05-06 18:02:18,2025-05-07 09:28:26,NaT,NaN,...,G004,Y,NaN,N,NaN,N,NaN,N,2024,강원특별자치도
978,CI,쿨사인,쿨사인,1600-4045,1,2025-04-30 02:46:42,2024-08-17 14:55:29,2024-08-17 16:27:15,NaT,NaN,...,G004,Y,NaN,N,NaN,N,NaN,N,2024,강원특별자치도


In [ ]:
# stat 칼럼 기반 필터링 함수 정의

def stat_filtering(df):
    return df[df['stat'].isin([1, 2, 3, 9])]

stat_filtering(all_data_4).shape

(90891, 35)

In [ ]:

# 현재 시점 기준 48시간 이내에 사용 기록이 있는지 여부 기반 필터링 함수 정의 

def recent_filtering(df, hours=48, now=None):
    
    if now is None:
        now = pd.Timestamp.now()
    
    # 최근 충전 '시작' 시간과 최근 충전 '종료' 시간 중 더 최근 시간을 기준으로 계산
    max_time = df[['lastTsdt', 'lastTedt']].max(axis=1)

    # 기준 시각에서 `hours` 이전보다 더 늦은 것만 남기기
    return df[max_time >= (now - pd.Timedelta(hours=hours))].copy()

In [ ]:
# 테스트트

from datetime import datetime
custom_now = pd.Timestamp(datetime(2025, 5, 16, 12, 0, 0))

recent_filtering(all_data_4, 48, now=custom_now).head(5).iloc[:, 10:]

,busiId,bnm,busiNm,busiCall,stat,statUpdDt,lastTsdt,lastTedt,nowTsdt,powerType,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
0,ME,환경부,환경부,1661-9408,2,2025-05-15 15:55:32,2025-05-15 14:53:26,2025-05-15 15:07:11,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
1,ME,환경부,환경부,1661-9408,2,2025-05-15 15:56:33,2025-05-15 13:33:14,2025-05-15 14:05:38,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
2,ME,환경부,환경부,1661-9408,2,2025-05-15 15:54:34,2025-05-14 19:45:11,2025-05-14 20:26:46,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
3,ME,환경부,환경부,1661-9408,2,2025-05-15 15:57:34,2025-05-15 11:15:42,2025-05-15 11:46:57,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
5,ME,환경부,환경부,1661-9408,2,2025-05-15 15:55:47,2025-05-14 20:58:55,2025-05-14 21:13:56,NaT,NaN,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도


In [ ]:
# 필터링 함수 통합한 최종 1차 필터링 함수 정의 

def filtering_first(df, hours=48, now=None):
    temp_df = stat_filtering(df)
    result_df = recent_filtering(temp_df, hours, now)
    return result_df

In [ ]:
filtering_first(all_data_4, 48, custom_now).shape

(38338, 35)

## 4. m km 기반 필터링

In [ ]:
# haversine 라이브러리를 이용해 계산 가능하긴 함. 그러나 계산 시간 이슈로 numpy 기반 구현 예정. 

# from haversine import haversine, Unit
# import pandas as pd


# def filter_by_distance(df, center=(37.5665, 126.9780), max_distance_km=5):
    
#     def calc_distance(row):
#         point = (row['lat'], row['lng'])
#         return haversine(center, point, unit=Unit.KILOMETERS)

#     df = df.copy()
#     df['distance_km'] = df.apply(calc_distance, axis=1)
#     return df[df['distance_km'] <= max_distance_km]


In [ ]:
import numpy as np

def filter_by_distance_vectorized(df, center, max_distance_km=5):
    R = 6371  # 지구 반지름 (단위: km)
    lat1, lon1 = np.radians(center)

    lat2 = np.radians(df['lat'].values)
    lon2 = np.radians(df['lng'].values)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distances = R * c

    return df.loc[distances <= max_distance_km].copy()


In [ ]:
filter_by_distance_vectorized(all_data_4, (36, 129), 5).head(5)

,statNm,statId,chgerId,chgerType,addr,addrDetail,location,useTime,lat,lng,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
164968,고경면사무소,ME18C157,1,6,경상북도 영천시 고경면 호국로 1065-4,NaN,NaN,24시간 이용가능,36.001288,129.046173,...,A002,Y,NaN,N,NaN,N,NaN,N,2018,경상북도
165270,고경면행정복지센터앞 공원 주차장,ME19F229,1,4,경상북도 영천시 고경면 호국로 1065-4,경상북도 영천시 고경면 호국로 1065-4,NaN,24시간 이용가능,36.001231,129.046306,...,B002,Y,NaN,N,NaN,N,NaN,N,2021,경상북도
165271,고경면행정복지센터앞 공원 주차장,ME19F229,2,4,경상북도 영천시 고경면 호국로 1065-4,경상북도 영천시 고경면 호국로 1065-4,NaN,24시간 이용가능,36.001231,129.046306,...,B002,Y,NaN,N,NaN,N,NaN,N,2019,경상북도
165812,망정2공영주차장,ME23A405,21,4,경상북도 영천시 망정동 417-21,NaN,NaN,24시간 이용가능,35.987394,128.957118,...,B001,Y,NaN,N,NaN,N,NaN,N,2023,경상북도
165813,망정2공영주차장,ME23A405,22,4,경상북도 영천시 망정동 417-21,NaN,NaN,24시간 이용가능,35.987394,128.957118,...,B001,Y,NaN,N,NaN,N,NaN,N,2023,경상북도


## 5. 로직 1 - 2차 필터링

2차 필터링: 사용자 입력 변수 기반 필터링
- output: 충전 용량(3, 7, 50, 100, 200)
- chgertype: 충전기 커넥터 유형(01:DC차데모,02: AC완속,03: DC차데모+AC3상,04: DC콤보,05: DC차데모+DC콤보, 06: DC차데모+AC3상+DC콤보, 07: AC3상, 08: DC콤보(완속), 09: NACS, 10: DC콤보+NACS)
- kind: 관련 시설 종류(공공시설, 주차시설 등)
- busid: 충전 사업자(GS 칼텍스, 현대자동차 등)

In [ ]:
def filter_by_user_input(df, 
                         output_values=None, 
                         chger_types=None, 
                         kinds=None, 
                         busi_ids=None):
    filtered = df.copy()

    if output_values is not None:
        filtered = filtered[filtered['output'].isin(output_values)]
    
    if chger_types is not None:
        filtered = filtered[filtered['chgerType'].astype(str).isin(chger_types)]

    if kinds is not None:
        filtered = filtered[filtered['kind'].isin(kinds)]
    
    if busi_ids is not None:
        filtered = filtered[filtered['busiId'].isin(busi_ids)]

    return filtered


In [ ]:
filter_by_user_input(
    df=all_data_4,
    output_values=[50, 100],
    busi_ids=['ME', 'GS']
).head(5)


,statNm,statId,chgerId,chgerType,addr,addrDetail,location,useTime,lat,lng,...,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year,regionName
0,원주(부산) 휴게소,ME178073,1,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
1,원주(부산) 휴게소,ME178073,2,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
2,원주(부산) 휴게소,ME178073,3,6,강원특별자치도 원주시 호저면 마근거리길 120 (옥산리) 211,NaN,NaN,24시간 이용가능,37.434578,127.929205,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
3,치악(부산) 휴게소,ME178077,1,6,강원특별자치도 원주시 신림면 치악로 416 (금창리),NaN,NaN,24시간 이용가능,37.253311,128.049451,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도
4,치악(부산) 휴게소,ME178077,2,6,강원특별자치도 원주시 신림면 치악로 416 (금창리),NaN,NaN,24시간 이용가능,37.253311,128.049451,...,C001,Y,NaN,N,NaN,N,NaN,N,2017,강원특별자치도


# 스코어링

로직 2: 충전소 스코어링

Note: 경로 좌표들을 받아서 N km 지점마다의 위치 좌표를 구하는 기능은 이후에 구현하고, 지금은 특정 위치 좌표가 주어졌을 때, m km 이내의 충전소 중 top k개를 추려내 반환하는 기능만 구현 예정.
고속도로인지 아닌지 확인할 수 있다고 전달받았는데, 그건 일단 이후에 고도화...
스코어링에 필요한 동적인 상태 변수를 받아와서 업데이트 하는 과정이 필요하지만, 일단은 서버에서 어떻게 돌아갈지 모르기 때문에 업데이트 된 상태라고 가정하고 구현하겠음.

Input: 충전소 추천이 필요한 지점의 위도 및 경도 (a, b)
Output: Input 지점에서 추천하는 충전소 top-k의 dataframe row

구현할 기능 1: input으로부터 m km 이내의 충전소 필터링
구현할 기능 2: 스코어링 식

스코어링에 활용할 변수:
1. 운영시간 ustime: 운영시간에 제한이 없는 충전소에 더 높은 점수 -> input 형식이 다양해서 전처리 필요함, 그리고 의미 있을지 잘 모르겠어서 고민 필요. 도착 예정 시간에 운영중인 충전소를 추천해주는 게 좋을 듯. 
경로 api 팀에 도착 예정 시각도 받아올 수 있는지 문의 -> 가능하면 time in ustime일 때 추천해주면 될 듯
2. 주차 무료 여부 parkingFree: 주차 무료 충전소에 더 높은 점수. 그런데 값이 없는 경우도 많아서...
3. 동적인 충전소 상태 변수: 
    1. 최근 사용 기록이 있는 충전소일수록 잘 작동하고 있다는 증거이므로 해당 충전소를 우선 추천("최근"의 정확한 기준을 정해야함. say 1일)
    2. 현재 시점(혹은 사용자의 예상 도착 시간 기준..?)의 충전소 혼잡 정도(충전중이지 않은 충전기 수/전체 충전기 수)를 계산하여, 충전소 혼잡도가 낮은, 사용이 원활한 충전소를 우선적으로 추천 -> 해야할까?
    3. 절대적으로 충전소의 퀄리티를 가늠하는 변수 - 정상 작동 충전기 수/비율 등을 기반으로 처리 -> 구체적 컬럼은 논의 필요
5. 비용 조건: 충전 요금(회원가 비회원가 평균 기준, 나중에 프론트에서만 둘 다 볼 수 있게 제공)을 기준으로 저렴한 충전소를 우선적으로 추천
6. 충전기 수: 해당 충전소의 충전기 수가 많은 충전소를 우선적으로 추천
7. 위치 가까운 순으로 높은 점수 부여, 위도 경도 기준으로 haversine library 불러와서 사용
8. 주위 편의시설 여부 : 존재한다면 우선 추천
9. 급속 / 완속 여부 -> 스코어링에도 넣을지 여부에 대한 논의 필요

스코어링 방법론:
WSM - 좋은 충전기 라벨링해서 머신러닝하는 방향으로 고려중

## 우선순위

1. 위치 가까운 순
2. 충전소 상태 - 최근 사용 기록, 고장/ 삭제 여부, 퀄리티